# 综合练习 2：用户订单行为综合分析

## 一、练习目标

本题的主要目标不是做复杂业务方案，而是训练前面五类题型的综合调用能力：

```text
01 Gap & Island：连续付费日期识别
02 Ranking：去重、最近记录、金额排名
03 Cumulative Analysis：累计支付金额
04 Time Comparison：上一次支付日期、间隔天数
05 Join and Subquery：用户表与订单表关联、缺失用户判断
```

本题要求分别完成：

```text
SQL 轨道
Pandas 轨道
```

当前阶段重点是语法熟练度和方法迁移，不要求写复杂业务结论。

---

## 二、业务背景

现在有两张表：

```text
df_users：用户信息表
df_orders：订单流水表
```

`df_users` 记录用户的基础信息。

`df_orders` 记录用户订单、支付状态、支付金额和更新时间。

由于数据来自真实业务系统，数据并不完全干净，存在：

```text
1. 订单状态大小写不统一；
2. 订单状态前后可能有空格；
3. 同一个 order_id 可能出现重复记录；
4. 有订单中的 user_id 不存在于用户表；
5. 有用户在用户表中存在，但没有任何订单；
6. 有订单未支付、取消、退款；
7. 部分城市字段缺失。
```

---

## 三、输入表

### 1. 用户表：df_users

字段：

```text
user_id
user_name
city
register_date
user_level
```

字段含义：

| 字段 | 含义 |
|---|---|
| user_id | 用户编号 |
| user_name | 用户名称 |
| city | 城市 |
| register_date | 注册日期 |
| user_level | 用户等级 |

---

### 2. 订单表：df_orders

字段：

```text
order_id
user_id
order_date
pay_date
order_status
order_amount
paid_amount
channel
updated_at
```

字段含义：

| 字段 | 含义 |
|---|---|
| order_id | 订单编号 |
| user_id | 用户编号 |
| order_date | 下单日期 |
| pay_date | 支付日期 |
| order_status | 订单状态 |
| order_amount | 下单金额 |
| paid_amount | 实际支付金额 |
| channel | 订单渠道 |
| updated_at | 记录更新时间 |

---

## 四、基础清洗规则

在正式分析前，需要先得到一张清洗后的订单表。

### 1. 订单状态标准化

原始 `order_status` 可能存在大小写和空格问题。

需要统一处理为：

```text
normalized_status = UPPER(TRIM(order_status))
```

例如：

| 原始值 | 标准化后 |
|---|---|
| `PAID` | `PAID` |
| ` paid ` | `PAID` |
| `Refunded` | `REFUNDED` |

---

### 2. 订单去重

同一个 `order_id` 可能出现多条记录。

去重规则：

```text
同一个 order_id，只保留 updated_at 最新的一条记录。
```

SQL 轨道建议使用：

```text
ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY updated_at DESC)
```

Pandas 轨道建议使用：

```text
sort_values() + drop_duplicates()
```

---

### 3. 有效支付订单定义

有效支付订单必须满足：

```text
normalized_status = 'PAID'
pay_date 不为空
paid_amount 不为空
```

只有有效支付订单才参与：

```text
支付金额统计
支付订单数统计
最近支付日期
连续支付日期
累计支付金额
```

---

## 五、输出结果一：付费订单明细表

请先生成一张付费订单明细表：

```text
df_paid_detail
```

每一行代表一笔有效支付订单。

输出字段：

```text
order_id
user_id
pay_date
paid_amount
paid_order_seq
previous_paid_date
days_from_previous_paid
running_paid_amount
```

字段规则：

### 1. paid_order_seq

用户内的第几笔有效支付订单。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
生成从 1 开始的序号。
```

SQL 轨道：

```text
ROW_NUMBER()
```

Pandas 轨道：

```text
sort_values() + groupby().cumcount() + 1
```

---

### 2. previous_paid_date

用户上一笔有效支付订单日期。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
取上一笔 pay_date。
```

SQL 轨道：

```text
LAG(pay_date)
```

Pandas 轨道：

```text
groupby().shift(1)
```

---

### 3. days_from_previous_paid

当前支付日期距离上一笔支付日期的天数。

规则：

```text
pay_date - previous_paid_date
```

第一笔支付订单没有上一笔支付日期，结果为空。

---

### 4. running_paid_amount

用户截至当前订单的累计支付金额。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
对 paid_amount 做累计求和。
```

SQL 轨道：

```text
SUM(paid_amount) OVER(
    PARTITION BY user_id
    ORDER BY pay_date, order_id
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

Pandas 轨道：

```text
groupby().cumsum()
```

---

## 六、输出结果二：用户付费汇总表

再生成一张用户级汇总表：

```text
df_user_paid_summary
```

每一行代表一个用户。

最终用户集合必须来自：

```text
df_users.user_id
+
df_orders.user_id
```

也就是说，最终结果既要包含用户表中的用户，也要包含订单表中出现但用户表缺失的未知用户。

输出字段：

```text
user_id
user_name
city
has_user_info
has_paid_order
paid_order_count
total_paid_amount
latest_paid_date
previous_paid_date
days_since_latest_paid
max_consecutive_paid_days
paid_amount_rank
user_tag
```

---

## 七、字段计算规则

### 1. has_user_info

表示该 `user_id` 是否存在于用户表 `df_users` 中。

规则：

```text
存在于 df_users：True
不存在于 df_users：False
```

用于识别：

```text
订单表中出现，但用户表中没有的未知用户。
```

---

### 2. has_paid_order

表示该用户是否存在有效支付订单。

规则：

```text
至少有一笔有效支付订单：True
否则：False
```

---

### 3. paid_order_count

有效支付订单数。

规则：

```text
按 user_id 分组，统计有效支付订单数量。
```

没有有效支付订单的用户记为 0。

---

### 4. total_paid_amount

累计有效支付金额。

规则：

```text
按 user_id 分组，对 paid_amount 求和。
```

没有有效支付订单的用户记为 0。

---

### 5. latest_paid_date

最近一次有效支付日期。

规则：

```text
按 user_id 分组，取 pay_date 最大值。
```

没有有效支付订单的用户为空。

---

### 6. previous_paid_date

最近一次支付之前的上一笔有效支付日期。

规则：

```text
先按 user_id 分组；
按 pay_date 降序排序；
取每个用户最近支付记录的上一笔支付日期。
```

可以用两种思路：

```text
思路一：
先在付费明细表中用 LAG / shift 得到 previous_paid_date，
再取每个用户最新一笔支付记录。

思路二：
按 pay_date 降序排序，用 ROW_NUMBER 找最近第 1 笔和第 2 笔。
```

---

### 7. days_since_latest_paid

以固定分析日期为基准，计算距离最近一次支付已经过去多少天。

分析日期固定为：

```text
2026-07-15
```

规则：

```text
analysis_date - latest_paid_date
```

没有有效支付订单的用户为空。

---

### 8. max_consecutive_paid_days

用户最长连续支付天数。

连续支付定义：

```text
同一个用户在自然日期上连续发生有效支付。
```

注意：

```text
同一用户同一天多笔支付，只算 1 个支付日。
```

例如：

| user_id | pay_date |
|---|---|
| U001 | 2026-07-01 |
| U001 | 2026-07-02 |
| U001 | 2026-07-04 |

最长连续支付天数为：

```text
2
```

因为 7 月 1 日和 7 月 2 日连续，7 月 4 日与前面断开。

没有有效支付订单的用户记为 0。

---

### 9. paid_amount_rank

按照 `total_paid_amount` 从高到低排名。

规则：

```text
金额相同使用并列排名。
```

SQL 轨道：

```text
RANK()
```

Pandas 轨道：

```text
rank(method='min', ascending=False)
```

---

### 10. user_tag

用户标签。

判断顺序从上到下：

| 条件 | user_tag |
|---|---|
| has_user_info = False | UNKNOWN_USER |
| has_paid_order = False | NO_PAID |
| total_paid_amount >= 500 and days_since_latest_paid <= 7 | HIGH_VALUE_ACTIVE |
| total_paid_amount >= 500 and days_since_latest_paid > 7 | HIGH_VALUE_SILENT |
| paid_order_count >= 3 | REPEAT_USER |
| 其他情况 | NORMAL_USER |

SQL 使用：

```text
CASE WHEN
```

Pandas 使用：

```text
np.select()
或多次 where / loc 赋值
```

---

## 八、本题必须覆盖的五类 Pattern

| 类别 | 本题中的使用点 |
|---|---|
| Gap & Island | 计算 max_consecutive_paid_days |
| Ranking | 订单去重、最近支付记录、paid_amount_rank |
| Cumulative Analysis | running_paid_amount |
| Time Comparison | previous_paid_date、days_from_previous_paid、days_since_latest_paid |
| Join and Subquery | 用户表和订单表关联、has_user_info、未知用户识别 |

---

## 九、完成要求

请分别完成：

```text
SQL 轨道
Pandas 轨道
```

当前可以先做 SQL 轨道，Pandas 轨道后续补。

最终至少输出两张表：

```text
df_paid_detail
df_user_paid_summary
```

要求：

```text
1. SQL 和 Pandas 的业务结果保持一致；
2. 中间表命名清晰；
3. 不要一口气写成一坨代码；
4. 每一步只解决一个明确问题；
5. 最终重点复盘 SQL / Pandas 双轨语法对应关系。
```

In [1106]:
import pandas as pd
import duckdb
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

analysis_date = pd.Timestamp("2026-07-15")

# =========================
# 用户表：df_users
# =========================

df_users = pd.DataFrame(
    [
        {
            "user_id": "U001",
            "user_name": "Alice",
            "city": "Beijing",
            "register_date": "2026-06-20",
            "user_level": "VIP",
        },
        {
            "user_id": "U002",
            "user_name": "Bob",
            "city": "Shanghai",
            "register_date": "2026-06-22",
            "user_level": "NORMAL",
        },
        {
            "user_id": "U003",
            "user_name": "Chen",
            "city": "Shenzhen",
            "register_date": "2026-06-25",
            "user_level": "VIP",
        },
        {
            "user_id": "U004",
            "user_name": "Diana",
            "city": None,          # 城市缺失
            "register_date": "2026-06-28",
            "user_level": "NORMAL",
        },
        {
            "user_id": "U005",
            "user_name": "Evan",
            "city": "Chengdu",
            "register_date": "2026-07-01",
            "user_level": "NORMAL",
        },
        {
            "user_id": "U006",
            "user_name": "Fang",
            "city": "Guangzhou",
            "register_date": "2026-07-02",
            "user_level": "VIP",
        },
        {
            "user_id": "U007",
            "user_name": "Gao",
            "city": "Wuhan",
            "register_date": "2026-07-03",
            "user_level": "NORMAL",
        },
        {
            "user_id": "U008",
            "user_name": "He",
            "city": "Xi'an",
            "register_date": "2026-07-05",
            "user_level": "NORMAL",
        },
        {
            "user_id": "U010",
            "user_name": "Ivy",
            "city": "Shanghai",
            "register_date": "2026-07-07",
            "user_level": "VIP",
        },
    ]
)

df_users["register_date"] = pd.to_datetime(df_users["register_date"])


# =========================
# 订单表：df_orders
# =========================

df_orders = pd.DataFrame(
    [
        # U001：多笔有效支付，7月1日和7月2日连续，7月10日最近支付
        {
            "order_id": "O1001",
            "user_id": "U001",
            "order_date": "2026-07-01",
            "pay_date": "2026-07-01",
            "order_status": "PAID",
            "order_amount": 120,
            "paid_amount": 120,
            "channel": "APP",
            "updated_at": "2026-07-01 09:10:00",
        },
        {
            "order_id": "O1002",
            "user_id": "U001",
            "order_date": "2026-07-02",
            "pay_date": "2026-07-02",
            "order_status": " paid ",   # 状态有空格和小写
            "order_amount": 180,
            "paid_amount": 180,
            "channel": "APP",
            "updated_at": "2026-07-02 10:20:00",
        },
        {
            "order_id": "O1003",
            "user_id": "U001",
            "order_date": "2026-07-10",
            "pay_date": "2026-07-10",
            "order_status": "PAID",
            "order_amount": 260,
            "paid_amount": 260,
            "channel": "WEB",
            "updated_at": "2026-07-10 18:30:00",
        },

        # U002：有支付，也有取消订单
        {
            "order_id": "O1004",
            "user_id": "U002",
            "order_date": "2026-07-01",
            "pay_date": "2026-07-01",
            "order_status": "PAID",
            "order_amount": 80,
            "paid_amount": 80,
            "channel": "APP",
            "updated_at": "2026-07-01 11:00:00",
        },
        {
            "order_id": "O1005",
            "user_id": "U002",
            "order_date": "2026-07-04",
            "pay_date": None,
            "order_status": "CANCELLED",
            "order_amount": 100,
            "paid_amount": None,
            "channel": "APP",
            "updated_at": "2026-07-04 12:00:00",
        },

        # U003：有重复订单 O1006，只保留 updated_at 最新的一条
        {
            "order_id": "O1006",
            "user_id": "U003",
            "order_date": "2026-07-03",
            "pay_date": "2026-07-03",
            "order_status": "PAID",
            "order_amount": 300,
            "paid_amount": 300,
            "channel": "WEB",
            "updated_at": "2026-07-03 09:00:00",
        },
        {
            "order_id": "O1006",
            "user_id": "U003",
            "order_date": "2026-07-03",
            "pay_date": "2026-07-03",
            "order_status": "PAID",
            "order_amount": 320,
            "paid_amount": 320,
            "channel": "WEB",
            "updated_at": "2026-07-03 10:30:00",  # 同订单较新记录
        },
        {
            "order_id": "O1007",
            "user_id": "U003",
            "order_date": "2026-07-12",
            "pay_date": "2026-07-12",
            "order_status": "PAID",
            "order_amount": 260,
            "paid_amount": 260,
            "channel": "APP",
            "updated_at": "2026-07-12 20:00:00",
        },

        # U004：只有退款和待支付，没有有效支付
        {
            "order_id": "O1008",
            "user_id": "U004",
            "order_date": "2026-07-02",
            "pay_date": "2026-07-02",
            "order_status": "Refunded",   # 状态大小写不统一
            "order_amount": 150,
            "paid_amount": 150,
            "channel": "WEB",
            "updated_at": "2026-07-03 08:00:00",
        },
        {
            "order_id": "O1009",
            "user_id": "U004",
            "order_date": "2026-07-06",
            "pay_date": None,
            "order_status": "PENDING",
            "order_amount": 200,
            "paid_amount": None,
            "channel": "APP",
            "updated_at": "2026-07-06 14:00:00",
        },

        # U005：连续 3 天有效支付，用于 Gap & Island
        {
            "order_id": "O1010",
            "user_id": "U005",
            "order_date": "2026-07-02",
            "pay_date": "2026-07-02",
            "order_status": "PAID",
            "order_amount": 40,
            "paid_amount": 40,
            "channel": "APP",
            "updated_at": "2026-07-02 09:00:00",
        },
        {
            "order_id": "O1011",
            "user_id": "U005",
            "order_date": "2026-07-03",
            "pay_date": "2026-07-03",
            "order_status": "PAID",
            "order_amount": 60,
            "paid_amount": 60,
            "channel": "APP",
            "updated_at": "2026-07-03 09:30:00",
        },
        {
            "order_id": "O1012",
            "user_id": "U005",
            "order_date": "2026-07-04",
            "pay_date": "2026-07-04",
            "order_status": "PAID",
            "order_amount": 70,
            "paid_amount": 70,
            "channel": "APP",
            "updated_at": "2026-07-04 10:00:00",
        },

        # U006：只有待支付，没有有效支付
        {
            "order_id": "O1013",
            "user_id": "U006",
            "order_date": "2026-07-08",
            "pay_date": None,
            "order_status": "PENDING",
            "order_amount": 500,
            "paid_amount": None,
            "channel": "WEB",
            "updated_at": "2026-07-08 16:00:00",
        },

        # U007：低金额有效支付
        {
            "order_id": "O1014",
            "user_id": "U007",
            "order_date": "2026-07-11",
            "pay_date": "2026-07-11",
            "order_status": "PAID",
            "order_amount": 50,
            "paid_amount": 50,
            "channel": "APP",
            "updated_at": "2026-07-11 13:00:00",
        },

        # U999：订单表中出现，但用户表中没有
        {
            "order_id": "O1015",
            "user_id": "U999",
            "order_date": "2026-07-05",
            "pay_date": "2026-07-05",
            "order_status": "PAID",
            "order_amount": 220,
            "paid_amount": 220,
            "channel": "WEB",
            "updated_at": "2026-07-05 15:00:00",
        },
        {
            "order_id": "O1016",
            "user_id": "U999",
            "order_date": "2026-07-06",
            "pay_date": "2026-07-06",
            "order_status": "PAID",
            "order_amount": 230,
            "paid_amount": 230,
            "channel": "WEB",
            "updated_at": "2026-07-06 15:30:00",
        },
    ]
)

date_cols = ["order_date", "pay_date", "updated_at"]

for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

df_users, df_orders

(  user_id user_name       city register_date user_level
 0    U001     Alice    Beijing    2026-06-20        VIP
 1    U002       Bob   Shanghai    2026-06-22     NORMAL
 2    U003      Chen   Shenzhen    2026-06-25        VIP
 3    U004     Diana        NaN    2026-06-28     NORMAL
 4    U005      Evan    Chengdu    2026-07-01     NORMAL
 5    U006      Fang  Guangzhou    2026-07-02        VIP
 6    U007       Gao      Wuhan    2026-07-03     NORMAL
 7    U008        He      Xi'an    2026-07-05     NORMAL
 8    U010       Ivy   Shanghai    2026-07-07        VIP,
    order_id user_id order_date   pay_date order_status  order_amount  paid_amount channel          updated_at
 0     O1001    U001 2026-07-01 2026-07-01         PAID           120        120.0     APP 2026-07-01 09:10:00
 1     O1002    U001 2026-07-02 2026-07-02        paid            180        180.0     APP 2026-07-02 10:20:00
 2     O1003    U001 2026-07-10 2026-07-10         PAID           260        260.0     WEB 2026-

### 1. 订单状态标准化

原始 `order_status` 可能存在大小写和空格问题。

需要统一处理为：

```text
normalized_status = UPPER(TRIM(order_status))
```

例如：

| 原始值 | 标准化后 |
|---|---|
| `PAID` | `PAID` |
| ` paid ` | `PAID` |
| `Refunded` | `REFUNDED` |


In [1107]:
# =============================
# SQL轨道（第一步：数据清洗）
# =============================

query_data_clean = '''

    SELECT
        order_id,
        user_id,
        order_date,
        pay_date,
        UPPER(TRIM(order_status)) AS order_status,
        order_amount,
        paid_amount,
        channel,
        updated_at
    FROM df_orders
    ORDER BY order_id      


'''
df_orders_clean = duckdb.execute(query_data_clean).fetchdf()
df_orders_clean

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,PAID,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1005,U002,2026-07-04,NaT,CANCELLED,100,NaN,APP,2026-07-04 12:00:00
5,O1006,U003,2026-07-03,2026-07-03,PAID,300,300.0,WEB,2026-07-03 09:00:00
6,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
7,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
8,O1008,U004,2026-07-02,2026-07-02,REFUNDED,150,150.0,WEB,2026-07-03 08:00:00
9,O1009,U004,2026-07-06,NaT,PENDING,200,NaN,APP,2026-07-06 14:00:00


In [1108]:
# =============================
# PANDAS轨道（第一步：数据清洗）
# =============================

df_orders_clean_pd = df_orders.copy()

df_orders_clean.loc[:,'order_status'].str.strip().str.upper()
df_orders_clean_pd

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,paid,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1005,U002,2026-07-04,NaT,CANCELLED,100,NaN,APP,2026-07-04 12:00:00
5,O1006,U003,2026-07-03,2026-07-03,PAID,300,300.0,WEB,2026-07-03 09:00:00
6,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
7,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
8,O1008,U004,2026-07-02,2026-07-02,Refunded,150,150.0,WEB,2026-07-03 08:00:00
9,O1009,U004,2026-07-06,NaT,PENDING,200,NaN,APP,2026-07-06 14:00:00


### 2. 订单去重

同一个 `order_id` 可能出现多条记录。

去重规则：

```text
同一个 order_id，只保留 updated_at 最新的一条记录。
```

In [1109]:
# ================================
# SQL轨道（第二步：保留最新订单信息）
# ================================

query_update_orders = '''

WITH date_rank AS(
    SELECT 
        order_id,
        user_id,
        order_date,
        pay_date,
        order_status,
        order_amount,
        paid_amount,
        channel,
        updated_at,
        ROW_NUMBER()
        OVER(
            PARTITION BY order_id ORDER BY updated_at DESC
        ) AS date_rank
    FROM df_orders_clean
    ORDER BY order_id,user_id
)

SELECT 
    order_id,
    user_id,
    order_date,
    pay_date,
    order_status,
    order_amount,
    paid_amount,
    channel,
    updated_at
FROM date_rank
WHERE date_rank = 1
ORDER BY order_id,user_id
'''
df_lastest_orders = duckdb.execute(query_update_orders).fetchdf()
df_lastest_orders

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,PAID,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1005,U002,2026-07-04,NaT,CANCELLED,100,NaN,APP,2026-07-04 12:00:00
5,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
6,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
7,O1008,U004,2026-07-02,2026-07-02,REFUNDED,150,150.0,WEB,2026-07-03 08:00:00
8,O1009,U004,2026-07-06,NaT,PENDING,200,NaN,APP,2026-07-06 14:00:00
9,O1010,U005,2026-07-02,2026-07-02,PAID,40,40.0,APP,2026-07-02 09:00:00


In [1110]:
# =================================
# Pandas轨道（第二步：保留最新订单信息）
# =================================

df_latest_orders_pd = (
    df_orders_clean
    .sort_values(
        by=['order_id', 'updated_at'],
        ascending=[True, False]
    )
    .assign(
        updated_at_rank=lambda x: (
            x.groupby('order_id')
            .cumcount()
            + 1
        )
    )
    .loc[lambda x: x['updated_at_rank'] == 1]
    .drop(columns='updated_at_rank')
    .reset_index(drop=True)
)

df_latest_orders_pd

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,PAID,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1005,U002,2026-07-04,NaT,CANCELLED,100,NaN,APP,2026-07-04 12:00:00
5,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
6,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
7,O1008,U004,2026-07-02,2026-07-02,REFUNDED,150,150.0,WEB,2026-07-03 08:00:00
8,O1009,U004,2026-07-06,NaT,PENDING,200,NaN,APP,2026-07-06 14:00:00
9,O1010,U005,2026-07-02,2026-07-02,PAID,40,40.0,APP,2026-07-02 09:00:00


### 3. 有效支付订单定义

有效支付订单必须满足：

```text
normalized_status = 'PAID'
pay_date 不为空
paid_amount 不为空
```

只有有效支付订单才参与：

```text
支付金额统计
支付订单数统计
最近支付日期
连续支付日期
累计支付金额
```

In [1111]:
# ================================
# SQL轨道（第三步：保留有效支付订单）
# ================================

query_valid_paid = '''

    SELECT
        order_id,
        user_id,
        order_date,
        pay_date,
        order_status,
        order_amount,
        paid_amount,
        channel,
        updated_at
    FROM df_latest_orders
    WHERE order_status = 'PAID'
    AND pay_date IS NOT NULL
    AND paid_amount IS NOT NULL
    ORDER BY order_id,user_id
'''
df_valid_paid = duckdb.execute(query_valid_paid).fetchdf()
df_valid_paid

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,PAID,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
5,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
6,O1010,U005,2026-07-02,2026-07-02,PAID,40,40.0,APP,2026-07-02 09:00:00
7,O1011,U005,2026-07-03,2026-07-03,PAID,60,60.0,APP,2026-07-03 09:30:00
8,O1012,U005,2026-07-04,2026-07-04,PAID,70,70.0,APP,2026-07-04 10:00:00
9,O1014,U007,2026-07-11,2026-07-11,PAID,50,50.0,APP,2026-07-11 13:00:00


In [1112]:
# ===================================
# PANDAS轨道（第三步：保留有效支付订单）
# ===================================

df_valid_paid_pd = (
    df_latest_orders_pd
    .loc[
        lambda x: (
            (x['order_status'] == 'PAID')
            & (x['pay_date'].notna())
            & (x['paid_amount'].notna())
        )
    ]
    .sort_values(by=['user_id', 'pay_date', 'order_id'])
    .reset_index(drop=True)
)

df_valid_paid_pd

,order_id,user_id,order_date,pay_date,order_status,order_amount,paid_amount,channel,updated_at
0,O1001,U001,2026-07-01,2026-07-01,PAID,120,120.0,APP,2026-07-01 09:10:00
1,O1002,U001,2026-07-02,2026-07-02,PAID,180,180.0,APP,2026-07-02 10:20:00
2,O1003,U001,2026-07-10,2026-07-10,PAID,260,260.0,WEB,2026-07-10 18:30:00
3,O1004,U002,2026-07-01,2026-07-01,PAID,80,80.0,APP,2026-07-01 11:00:00
4,O1006,U003,2026-07-03,2026-07-03,PAID,320,320.0,WEB,2026-07-03 10:30:00
5,O1007,U003,2026-07-12,2026-07-12,PAID,260,260.0,APP,2026-07-12 20:00:00
6,O1010,U005,2026-07-02,2026-07-02,PAID,40,40.0,APP,2026-07-02 09:00:00
7,O1011,U005,2026-07-03,2026-07-03,PAID,60,60.0,APP,2026-07-03 09:30:00
8,O1012,U005,2026-07-04,2026-07-04,PAID,70,70.0,APP,2026-07-04 10:00:00
9,O1014,U007,2026-07-11,2026-07-11,PAID,50,50.0,APP,2026-07-11 13:00:00


## 五、输出结果一：付费订单明细表

请先生成一张付费订单明细表：

```text
df_paid_detail
```

每一行代表一笔有效支付订单。

输出字段：

```text
order_id
user_id
pay_date
paid_amount
paid_order_seq
previous_paid_date
days_from_previous_paid
running_paid_amount
```

字段规则：

### 1. paid_order_seq

用户内的第几笔有效支付订单。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
生成从 1 开始的序号。
```
SQL 轨道：

```text
ROW_NUMBER()
```

Pandas 轨道：

```text
sort_values() + groupby().cumcount() + 1
```

---

### 2. previous_paid_date

用户上一笔有效支付订单日期。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
取上一笔 pay_date。
```

SQL 轨道：

```text
LAG(pay_date)
```

Pandas 轨道：

```text
groupby().shift(1)
```

---

### 3. days_from_previous_paid

当前支付日期距离上一笔支付日期的天数。

规则：

```text
pay_date - previous_paid_date
```

第一笔支付订单没有上一笔支付日期，结果为空。

---

### 4. running_paid_amount

用户截至当前订单的累计支付金额。

规则：

```text
按 user_id 分组；
按 pay_date、order_id 升序排序；
对 paid_amount 做累计求和。
```

SQL 轨道：

```text
SUM(paid_amount) OVER(
    PARTITION BY user_id
    ORDER BY pay_date, order_id
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

Pandas 轨道：

```text
groupby().cumsum()
```

---


In [1113]:
# ====================================================
# SQL轨道（第四步：付费订单明细表）
# ====================================================

query_valid_orders_rank = '''

WITH rank_table AS(

SELECT
    order_id,
    user_id,
    pay_date,
    paid_amount,
    ROW_NUMBER()
    OVER(PARTITION BY user_id ORDER BY pay_date,order_id ) AS paid_order_seq,
    LAG(pay_date)
    OVER(PARTITION BY user_id ORDER BY pay_date,order_id) AS previous_paid_date,
FROM df_valid_paid
)
SELECT
    order_id,
    user_id,
    pay_date,
    paid_amount,
    paid_order_seq,
    previous_paid_date,
    pay_date - previous_paid_date AS days_from_previous_paid,
    SUM(paid_amount)
    OVER(
        PARTITION BY user_id ORDER BY pay_date,order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW 
        ) AS running_paid_amount
FROM rank_table
ORDER BY order_id,user_id
'''
df_valid_order_info = duckdb.execute(query_valid_orders_rank).fetchdf()
df_valid_order_info

,order_id,user_id,pay_date,paid_amount,paid_order_seq,previous_paid_date,days_from_previous_paid,running_paid_amount
0,O1001,U001,2026-07-01,120.0,1,NaT,NaT,120.0
1,O1002,U001,2026-07-02,180.0,2,2026-07-01,1 days,300.0
2,O1003,U001,2026-07-10,260.0,3,2026-07-02,8 days,560.0
3,O1004,U002,2026-07-01,80.0,1,NaT,NaT,80.0
4,O1006,U003,2026-07-03,320.0,1,NaT,NaT,320.0
5,O1007,U003,2026-07-12,260.0,2,2026-07-03,9 days,580.0
6,O1010,U005,2026-07-02,40.0,1,NaT,NaT,40.0
7,O1011,U005,2026-07-03,60.0,2,2026-07-02,1 days,100.0
8,O1012,U005,2026-07-04,70.0,3,2026-07-03,1 days,170.0
9,O1014,U007,2026-07-11,50.0,1,NaT,NaT,50.0


In [1114]:
# ====================================================
# PANDAS轨道（第四步：付费订单明细表）
# ====================================================

df_valid_order_info_pd = (

    df_valid_paid_pd
    .sort_values(by=['user_id','pay_date','order_id'])
    .assign(
        paid_order_seq = lambda x:(
            x.groupby('user_id')
            .cumcount() + 1
        ),
        previous_paid_date = lambda x:(
            x.groupby('user_id')['pay_date']
            .shift(1)
        ),
        days_from_previous_paid = lambda x:(
            x['pay_date'] - x['previous_paid_date']
        ).dt.days.astype('Int64'),
        running_paid_amount = lambda x:(
            x.groupby('user_id')['paid_amount']
            .cumsum()
        )
    )
    [
        [
            'user_id',
            'order_id',
            'pay_date',
            'paid_amount',
            'paid_order_seq',
            'previous_paid_date',
            'days_from_previous_paid',
            'running_paid_amount'
        ]
    ]
    .sort_values(by=['user_id','pay_date','order_id'])
    .reset_index(drop=True)
    
)
df_valid_order_info_pd

,user_id,order_id,pay_date,paid_amount,paid_order_seq,previous_paid_date,days_from_previous_paid,running_paid_amount
0,U001,O1001,2026-07-01,120.0,1,NaT,<NA>,120.0
1,U001,O1002,2026-07-02,180.0,2,2026-07-01,1,300.0
2,U001,O1003,2026-07-10,260.0,3,2026-07-02,8,560.0
3,U002,O1004,2026-07-01,80.0,1,NaT,<NA>,80.0
4,U003,O1006,2026-07-03,320.0,1,NaT,<NA>,320.0
5,U003,O1007,2026-07-12,260.0,2,2026-07-03,9,580.0
6,U005,O1010,2026-07-02,40.0,1,NaT,<NA>,40.0
7,U005,O1011,2026-07-03,60.0,2,2026-07-02,1,100.0
8,U005,O1012,2026-07-04,70.0,3,2026-07-03,1,170.0
9,U007,O1014,2026-07-11,50.0,1,NaT,<NA>,50.0


## 六、输出结果二：用户付费汇总表

再生成一张用户级汇总表：

```text
df_user_paid_summary
```

每一行代表一个用户。

最终用户集合必须来自：

```text
df_users.user_id
+
df_orders.user_id
```

也就是说，最终结果既要包含用户表中的用户，也要包含订单表中出现但用户表缺失的未知用户。

输出字段：

```text
user_id
user_name
city
has_user_info
has_paid_order
paid_order_count
total_paid_amount
latest_paid_date
previous_paid_date
days_since_latest_paid
max_consecutive_paid_days
paid_amount_rank
user_tag

In [1115]:
# ====================================================
# SQL轨道（第五步：df_users中不存在的ID）
# ====================================================

query_paid_summary = '''

WITH user_id_summary AS(

    SELECT
        user_id
    FROM df_users

    UNION

    SELECT
        user_id
    FROM df_orders
)
SELECT
    ds.user_id,
    du.user_name,
    du.city,
    CASE
        WHEN EXISTS(
            SELECT
                1
            FROM df_users AS u
            WHERE u.user_id = ds.user_id
        )
        THEN True
        ELSE False
    END AS has_user_info
FROM user_id_summary AS ds
LEFT JOIN df_users AS du
ON ds.user_id = du.user_id
ORDER BY user_id
'''
df_user_base = duckdb.execute(query_paid_summary).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info
0,U001,Alice,Beijing,True
1,U002,Bob,Shanghai,True
2,U003,Chen,Shenzhen,True
3,U004,Diana,NaN,True
4,U005,Evan,Chengdu,True
5,U006,Fang,Guangzhou,True
6,U007,Gao,Wuhan,True
7,U008,He,Xi'an,True
8,U010,Ivy,Shanghai,True
9,U999,NaN,NaN,False


In [1116]:
# ====================================================
# PANDAS轨道（第五步：df_users中不存在的ID）
# ====================================================

df_user_id_collect_pd = (

    pd.concat(
        [
        df_orders[['user_id']],
        df_users[['user_id']]
        ],
        ignore_index = True
    )
    .drop_duplicates()
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
df_user_base_pd = (

    df_user_id_collect_pd
    .merge(
        df_users,
        how='left',
        on='user_id'
    )
    .assign(
        has_user_info = lambda x:(
            x['user_id'].isin(df_users['user_id'])
        )
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info'
        ]
    ]
    .sort_values(by='user_id')
    .reset_index(drop=True)
    
)
df_user_base_pd

,user_id,user_name,city,has_user_info
0,U001,Alice,Beijing,True
1,U002,Bob,Shanghai,True
2,U003,Chen,Shenzhen,True
3,U004,Diana,NaN,True
4,U005,Evan,Chengdu,True
5,U006,Fang,Guangzhou,True
6,U007,Gao,Wuhan,True
7,U008,He,Xi'an,True
8,U010,Ivy,Shanghai,True
9,U999,NaN,NaN,False


### 2. has_paid_order

表示该用户是否存在有效支付订单。

规则：

```text
至少有一笔有效支付订单：True
否则：False
```

---

In [1117]:
# ====================================================
# SQL轨道（第六步：是否存在有效支付订单）
# ====================================================

query_is_valid_paid = '''

    SELECT
        db.user_id,
        db.user_name,
        db.city,
        db.has_user_info,
        CASE
            WHEN EXISTS (
                SELECT
                    1
                FROM df_valid_paid AS dvp
                WHERE dvp.user_id = db.user_id
            )
            THEN True
            ELSE False
        END AS has_paid_order

    FROM df_user_base AS db
    ORDER BY db.user_id
'''
df_user_base = duckdb.execute(query_is_valid_paid).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info,has_paid_order
0,U001,Alice,Beijing,True,True
1,U002,Bob,Shanghai,True,True
2,U003,Chen,Shenzhen,True,True
3,U004,Diana,NaN,True,False
4,U005,Evan,Chengdu,True,True
5,U006,Fang,Guangzhou,True,False
6,U007,Gao,Wuhan,True,True
7,U008,He,Xi'an,True,False
8,U010,Ivy,Shanghai,True,False
9,U999,NaN,NaN,False,True


In [1118]:
# ====================================================
# PANDAS轨道（第六步：是否存在有效支付订单）
# ====================================================

df_user_base_pd = (
    df_user_base_pd
    .assign(
        has_paid_order = lambda x:(
            x['user_id'].isin(df_valid_paid_pd['user_id'])
        )
    )
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order
0,U001,Alice,Beijing,True,True
1,U002,Bob,Shanghai,True,True
2,U003,Chen,Shenzhen,True,True
3,U004,Diana,NaN,True,False
4,U005,Evan,Chengdu,True,True
5,U006,Fang,Guangzhou,True,False
6,U007,Gao,Wuhan,True,True
7,U008,He,Xi'an,True,False
8,U010,Ivy,Shanghai,True,False
9,U999,NaN,NaN,False,True


### 3. paid_order_count

有效支付订单数。

规则：

```text
按 user_id 分组，统计有效支付订单数量。
```

没有有效支付订单的用户记为 0。


In [1119]:
# ====================================================
# SQL轨道（第七步：有效支付的订单数量）
# ====================================================

query_valid_paid_count = '''

WITH valid_paid_count AS(

    SELECT
        user_id,
        COUNT(order_id) AS paid_order_count

    FROM df_valid_paid
    GROUP BY user_id
)
SELECT
    dub.user_id,
    dub.user_name,
    dub.city,
    dub.has_user_info,
    dub.has_paid_order,
    COALESCE(
        vpc.paid_order_count,0
    ) AS paid_order_count
FROM df_user_base AS dub
LEFT JOIN valid_paid_count AS vpc
ON dub.user_id = vpc.user_id
ORDER BY dub.user_id
'''

df_user_base  = duckdb.execute(query_valid_paid_count).fetchdf()
df_user_base 

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count
0,U001,Alice,Beijing,True,True,3
1,U002,Bob,Shanghai,True,True,1
2,U003,Chen,Shenzhen,True,True,2
3,U004,Diana,NaN,True,False,0
4,U005,Evan,Chengdu,True,True,3
5,U006,Fang,Guangzhou,True,False,0
6,U007,Gao,Wuhan,True,True,1
7,U008,He,Xi'an,True,False,0
8,U010,Ivy,Shanghai,True,False,0
9,U999,NaN,NaN,False,True,2


In [1120]:
# ====================================================
# PANDAS轨道（第七步：有效支付的订单数量）
# ====================================================

df_user_base_pd = (

    df_valid_paid_pd
    .groupby('user_id',as_index = False)
    .agg(
        paid_order_count=('order_id','count')
    )
    .merge(
        df_user_base_pd,
        how='right',
        on='user_id'
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info',
            'has_paid_order',
            'paid_order_count'
        ]
    ]
    .assign(
        paid_order_count = lambda x:x['paid_order_count'].fillna(0).astype('Int64')
    )
    .sort_values(by='user_id')
    .reset_index(drop=True)
    
)

df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count
0,U001,Alice,Beijing,True,True,3
1,U002,Bob,Shanghai,True,True,1
2,U003,Chen,Shenzhen,True,True,2
3,U004,Diana,NaN,True,False,0
4,U005,Evan,Chengdu,True,True,3
5,U006,Fang,Guangzhou,True,False,0
6,U007,Gao,Wuhan,True,True,1
7,U008,He,Xi'an,True,False,0
8,U010,Ivy,Shanghai,True,False,0
9,U999,NaN,NaN,False,True,2


### 4. total_paid_amount

累计有效支付金额。

规则：

```text
按 user_id 分组，对 paid_amount 求和。
```

没有有效支付订单的用户记为 0。

In [1121]:
# ====================================================
# SQL轨道（第八步：累计有效支付金额）
# ====================================================

query_valid_paid_sum = '''

WITH total_paid AS(

    SELECT
        user_id,
        SUM(paid_amount) AS total_paid_amount

    FROM df_valid_paid
    GROUP BY user_id
)
SELECT
    dub.user_id,
    dub.user_name,
    dub.city,
    dub.has_user_info,
    dub.has_paid_order,
    dub.paid_order_count,
    COALESCE(
        tp.total_paid_amount,0
    ) AS total_paid_amount
FROM df_user_base AS dub
LEFT JOIN total_paid AS tp
ON dub.user_id = tp.user_id
ORDER BY dub.user_id
'''

df_user_base  = duckdb.execute(query_valid_paid_sum).fetchdf()
df_user_base 


,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount
0,U001,Alice,Beijing,True,True,3,560.0
1,U002,Bob,Shanghai,True,True,1,80.0
2,U003,Chen,Shenzhen,True,True,2,580.0
3,U004,Diana,NaN,True,False,0,0.0
4,U005,Evan,Chengdu,True,True,3,170.0
5,U006,Fang,Guangzhou,True,False,0,0.0
6,U007,Gao,Wuhan,True,True,1,50.0
7,U008,He,Xi'an,True,False,0,0.0
8,U010,Ivy,Shanghai,True,False,0,0.0
9,U999,NaN,NaN,False,True,2,450.0


In [1122]:
# ====================================================
# PANDAS轨道（第八步：累计有效支付金额）
# ====================================================

df_valid_amount_sum_pd = (

    df_valid_paid_pd
    .groupby('user_id',as_index=False)
    .agg(
        total_paid_amount=('paid_amount','sum')
    )
)

df_user_base_pd = (
    df_user_base_pd
    .merge(
        df_valid_amount_sum_pd,
        how='left',
        on='user_id'
    )
    .assign(
        total_paid_amount = lambda x:(
            x['total_paid_amount'].fillna(0).astype('Int64')
        )
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info',
            'has_paid_order',
            'paid_order_count',
            'total_paid_amount'
        ]
    ]
    .sort_values(by='user_id')
    .reset_index(drop=True)
    
)
df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount
0,U001,Alice,Beijing,True,True,3,560
1,U002,Bob,Shanghai,True,True,1,80
2,U003,Chen,Shenzhen,True,True,2,580
3,U004,Diana,NaN,True,False,0,0
4,U005,Evan,Chengdu,True,True,3,170
5,U006,Fang,Guangzhou,True,False,0,0
6,U007,Gao,Wuhan,True,True,1,50
7,U008,He,Xi'an,True,False,0,0
8,U010,Ivy,Shanghai,True,False,0,0
9,U999,NaN,NaN,False,True,2,450


### 5. latest_paid_date

最近一次有效支付日期。

规则：

```text
按 user_id 分组，取 pay_date 最大值。
```

没有有效支付订单的用户为空。


In [1123]:
# ====================================================
# SQL轨道（第九步：最近一次有效支付日期）
# ====================================================

query_valid_paid_latest_date = '''

WITH latest_paid AS(

    SELECT
        user_id,
        MAX(pay_date) AS latest_paid_date

    FROM df_valid_paid
    GROUP BY user_id
)
SELECT
    dub.user_id,
    dub.user_name,
    dub.city,
    dub.has_user_info,
    dub.has_paid_order,
    dub.paid_order_count,
    dub.total_paid_amount,
    lp.latest_paid_date

FROM df_user_base AS dub
LEFT JOIN latest_paid AS lp
ON dub.user_id = lp.user_id
ORDER BY dub.user_id
'''

df_user_base  = duckdb.execute(query_valid_paid_latest_date).fetchdf()
df_user_base 




,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date
0,U001,Alice,Beijing,True,True,3,560.0,2026-07-10
1,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01
2,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12
3,U004,Diana,NaN,True,False,0,0.0,NaT
4,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04
5,U006,Fang,Guangzhou,True,False,0,0.0,NaT
6,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11
7,U008,He,Xi'an,True,False,0,0.0,NaT
8,U010,Ivy,Shanghai,True,False,0,0.0,NaT
9,U999,NaN,NaN,False,True,2,450.0,2026-07-06


In [1124]:
# ====================================================
# PANDAS轨道（第九步：最近一次有效支付日期）
# ====================================================

df_latest_valid_paid_date_pd = (

    df_valid_paid_pd
    
    .groupby('user_id',as_index=False)
    .agg(
        latest_paid_date=('pay_date','max')
    )
)
df_user_base_pd = (
    df_user_base_pd
    .merge(
        df_latest_valid_paid_date_pd,
        how='left',
        on='user_id'
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info',
            'has_paid_order',
            'paid_order_count',
            'total_paid_amount',
            'latest_paid_date'
        ]
    ]
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date
0,U001,Alice,Beijing,True,True,3,560,2026-07-10
1,U002,Bob,Shanghai,True,True,1,80,2026-07-01
2,U003,Chen,Shenzhen,True,True,2,580,2026-07-12
3,U004,Diana,NaN,True,False,0,0,NaT
4,U005,Evan,Chengdu,True,True,3,170,2026-07-04
5,U006,Fang,Guangzhou,True,False,0,0,NaT
6,U007,Gao,Wuhan,True,True,1,50,2026-07-11
7,U008,He,Xi'an,True,False,0,0,NaT
8,U010,Ivy,Shanghai,True,False,0,0,NaT
9,U999,NaN,NaN,False,True,2,450,2026-07-06


### 6. previous_paid_date

最近一次支付之前的上一笔有效支付日期。

规则：

```text
先按 user_id 分组；
按 pay_date 降序排序；
取每个用户最近支付记录的上一笔支付日期。
```

In [1125]:
# ====================================================
# SQL轨道（第十步：最近一次有效支付日期的上一笔支付日期）
# ====================================================

query_previous_pay_date = '''

WITH previous_pay_date AS(

    SELECT
        user_id,
        pay_date,
        order_id,
        LAG(pay_date)
        OVER(PARTITION BY user_id ORDER BY pay_date,order_id ) AS previous_paid_date

    FROM df_valid_paid
),
pay_date_rank AS(
    SELECT
        user_id,
        pay_date,
        previous_paid_date,
        ROW_NUMBER()
        OVER(PARTITION BY user_id ORDER BY pay_date DESC,order_id DESC ) AS date_rank
    FROM previous_pay_date
),
latest_date AS(
    SELECT 
        user_id,
        previous_paid_date
    FROM pay_date_rank
    WHERE date_rank = 1
)
SELECT 
    dub.user_id,
    dub.user_name,
    dub.city,
    dub.has_user_info,
    dub.has_paid_order,
    dub.paid_order_count,
    dub.total_paid_amount,
    dub.latest_paid_date,
    ld.previous_paid_date

FROM df_user_base AS dub
LEFT JOIN latest_date AS ld
    ON dub.user_id = ld.user_id
ORDER BY dub.user_id

'''
df_user_base = duckdb.execute(query_previous_pay_date).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date
0,U001,Alice,Beijing,True,True,3,560.0,2026-07-10,2026-07-02
1,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01,NaT
2,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12,2026-07-03
3,U004,Diana,NaN,True,False,0,0.0,NaT,NaT
4,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04,2026-07-03
5,U006,Fang,Guangzhou,True,False,0,0.0,NaT,NaT
6,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11,NaT
7,U008,He,Xi'an,True,False,0,0.0,NaT,NaT
8,U010,Ivy,Shanghai,True,False,0,0.0,NaT,NaT
9,U999,NaN,NaN,False,True,2,450.0,2026-07-06,2026-07-05


In [1126]:
# ====================================================
# PANDAS轨道（第十步：最近一次有效支付日期的上一笔支付日期）
# ====================================================

df_user_base_pd= (

    df_valid_paid_pd

    .sort_values(by=['user_id','pay_date','order_id'])
    .assign(
        previous_paid_date=lambda x: (
            x.groupby('user_id')['pay_date']
            .shift(1)
        )
    )
    .sort_values(by=['user_id','pay_date','order_id'],ascending=[True,False,False])
    .assign(
        date_rank = lambda x:(
            x.groupby('user_id')['pay_date']
            .cumcount() + 1
        )
    )
    .loc[lambda x:x['date_rank'] == 1,['user_id','previous_paid_date']]
    .merge(
        df_user_base_pd,
        how='right',
        on='user_id'
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info',
            'has_paid_order',
            'paid_order_count',
            'total_paid_amount',
            'latest_paid_date',
            'previous_paid_date'

        ]
    ]
    .sort_values(by='user_id')
    .reset_index(drop=True)
    
)
df_user_base_pd


,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date
0,U001,Alice,Beijing,True,True,3,560,2026-07-10,2026-07-02
1,U002,Bob,Shanghai,True,True,1,80,2026-07-01,NaT
2,U003,Chen,Shenzhen,True,True,2,580,2026-07-12,2026-07-03
3,U004,Diana,NaN,True,False,0,0,NaT,NaT
4,U005,Evan,Chengdu,True,True,3,170,2026-07-04,2026-07-03
5,U006,Fang,Guangzhou,True,False,0,0,NaT,NaT
6,U007,Gao,Wuhan,True,True,1,50,2026-07-11,NaT
7,U008,He,Xi'an,True,False,0,0,NaT,NaT
8,U010,Ivy,Shanghai,True,False,0,0,NaT,NaT
9,U999,NaN,NaN,False,True,2,450,2026-07-06,2026-07-05


### 7. days_since_latest_paid

以固定分析日期为基准，计算距离最近一次支付已经过去多少天。

分析日期固定为：

```text
2026-07-15
```

规则：

```text
analysis_date - latest_paid_date
```

没有有效支付订单的用户为空。

In [1127]:
# ======================================================
# SQL轨道（第十一步：2026-07-15离最近一次有效支付日期的天数）
# ======================================================

query_duration = '''

SELECT
    *,
    date_diff('day',latest_paid_date,DATE '2026-07-15') AS days_since_latest_paid
FROM df_user_base
ORDER BY user_id
'''
df_user_base = duckdb.execute(query_duration).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid
0,U001,Alice,Beijing,True,True,3,560.0,2026-07-10,2026-07-02,5
1,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01,NaT,14
2,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12,2026-07-03,3
3,U004,Diana,NaN,True,False,0,0.0,NaT,NaT,<NA>
4,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04,2026-07-03,11
5,U006,Fang,Guangzhou,True,False,0,0.0,NaT,NaT,<NA>
6,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11,NaT,4
7,U008,He,Xi'an,True,False,0,0.0,NaT,NaT,<NA>
8,U010,Ivy,Shanghai,True,False,0,0.0,NaT,NaT,<NA>
9,U999,NaN,NaN,False,True,2,450.0,2026-07-06,2026-07-05,9


In [1128]:
# =========================================================
# PANDAS轨道（第十一步：2026-07-15离最近一次有效支付日期的天数）
# =========================================================

df_user_base_pd = (

    df_user_base_pd
    .assign(
        days_since_latest_paid = lambda x:(
            (pd.Timestamp('2026-07-15')-x['latest_paid_date']).dt.days.astype('Int64')
        )
    )
    .sort_values(by='user_id')
    .reset_index(drop=True)
)
df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid
0,U001,Alice,Beijing,True,True,3,560,2026-07-10,2026-07-02,5
1,U002,Bob,Shanghai,True,True,1,80,2026-07-01,NaT,14
2,U003,Chen,Shenzhen,True,True,2,580,2026-07-12,2026-07-03,3
3,U004,Diana,NaN,True,False,0,0,NaT,NaT,<NA>
4,U005,Evan,Chengdu,True,True,3,170,2026-07-04,2026-07-03,11
5,U006,Fang,Guangzhou,True,False,0,0,NaT,NaT,<NA>
6,U007,Gao,Wuhan,True,True,1,50,2026-07-11,NaT,4
7,U008,He,Xi'an,True,False,0,0,NaT,NaT,<NA>
8,U010,Ivy,Shanghai,True,False,0,0,NaT,NaT,<NA>
9,U999,NaN,NaN,False,True,2,450,2026-07-06,2026-07-05,9


### 8. max_consecutive_paid_days

用户最长连续支付天数。

连续支付定义：

```text
同一个用户在自然日期上连续发生有效支付。
```

注意：

```text
同一用户同一天多笔支付，只算 1 个支付日。
```

In [1129]:
# ======================================================
# SQL轨道（第十二步：用户最长连续支付天数）
# ======================================================

query_consecutive_paid_days = '''

WITH rank_table AS (

    SELECT
        user_id,
        order_id,
        pay_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, pay_date
            ORDER BY order_id
        ) AS order_rank
    FROM df_valid_paid
),

paid_days AS (

    SELECT 
        user_id,
        order_id,
        pay_date
    FROM rank_table
    WHERE order_rank = 1
),

previous_date AS (

    SELECT
        user_id,
        order_id,
        pay_date,
        LAG(pay_date) OVER (
            PARTITION BY user_id
            ORDER BY pay_date
        ) AS previous_paid_date
    FROM paid_days
),

consecutive_start AS (

    SELECT
        user_id,
        order_id,
        pay_date,
        previous_paid_date,
        CASE
            WHEN previous_paid_date IS NULL
              OR pay_date > previous_paid_date + INTERVAL 1 DAY
            THEN 1
            ELSE 0
        END AS consecutive_start_sign
    FROM previous_date
),

phase_table AS (

    SELECT 
        *,
        SUM(consecutive_start_sign) OVER (
            PARTITION BY user_id
            ORDER BY pay_date
        )::INTEGER AS phase_sign
    FROM consecutive_start
),

consecutive_count AS (

    SELECT 
        user_id,
        phase_sign,
        COUNT(*)::INTEGER AS consecutive_count
    FROM phase_table
    GROUP BY user_id, phase_sign
),

max_consecutive_days AS (

    SELECT
        user_id,
        MAX(consecutive_count)::INTEGER AS max_consecutive_paid_days
    FROM consecutive_count
    GROUP BY user_id
)

SELECT 
    dub.*,
    COALESCE(mcd.max_consecutive_paid_days, 0)::INTEGER AS max_consecutive_paid_days
FROM df_user_base AS dub
LEFT JOIN max_consecutive_days AS mcd
    ON dub.user_id = mcd.user_id
ORDER BY dub.user_id

'''

df_user_base = duckdb.execute(query_consecutive_paid_days).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid,max_consecutive_paid_days
0,U001,Alice,Beijing,True,True,3,560.0,2026-07-10,2026-07-02,5,2
1,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01,NaT,14,1
2,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12,2026-07-03,3,1
3,U004,Diana,NaN,True,False,0,0.0,NaT,NaT,<NA>,0
4,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04,2026-07-03,11,3
5,U006,Fang,Guangzhou,True,False,0,0.0,NaT,NaT,<NA>,0
6,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11,NaT,4,1
7,U008,He,Xi'an,True,False,0,0.0,NaT,NaT,<NA>,0
8,U010,Ivy,Shanghai,True,False,0,0.0,NaT,NaT,<NA>,0
9,U999,NaN,NaN,False,True,2,450.0,2026-07-06,2026-07-05,9,2


In [1130]:
# ======================================================
# PANDAS轨道（第十二步：用户最长连续支付天数）
# ======================================================

df_previous_pay_date = (

    df_valid_paid_pd
    [['user_id', 'pay_date']]
    .drop_duplicates()
    .sort_values(by=['user_id', 'pay_date'])
    .assign(
        previous_paid_date=lambda x: (
            x.groupby('user_id')['pay_date']
            .shift(1)
        )
    )
    .assign(
        consecutive_start_sign=lambda x: (
            (x['previous_paid_date'].isna())
            | (
                x['pay_date']
                > x['previous_paid_date'] + pd.Timedelta(days=1)
            )
        ).astype(int),
        phase_sign=lambda x: (
            x.groupby('user_id')['consecutive_start_sign']
            .cumsum()
        )
    )
)

df_consecutive_paid_days_pd = (

    df_previous_pay_date
    .groupby(['user_id', 'phase_sign'], as_index=False)
    .agg(
        consecutive_paid_days=('pay_date', 'count')
    )
    .groupby('user_id', as_index=False)
    .agg(
        max_consecutive_paid_days=('consecutive_paid_days', 'max')
    )
)

df_user_base_pd = (

    df_user_base_pd
    .merge(
        df_consecutive_paid_days_pd,
        how='left',
        on='user_id'
    )
    .assign(
        max_consecutive_paid_days=lambda x: (
            x['max_consecutive_paid_days']
            .fillna(0)
            .astype('Int64')
        )
    )
    [
        [
            'user_id',
            'user_name',
            'city',
            'has_user_info',
            'has_paid_order',
            'paid_order_count',
            'total_paid_amount',
            'latest_paid_date',
            'previous_paid_date',
            'days_since_latest_paid',
            'max_consecutive_paid_days'
        ]
    ]
    .sort_values(by='user_id')
    .reset_index(drop=True)

)

df_user_base_pd

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid,max_consecutive_paid_days
0,U001,Alice,Beijing,True,True,3,560,2026-07-10,2026-07-02,5,2
1,U002,Bob,Shanghai,True,True,1,80,2026-07-01,NaT,14,1
2,U003,Chen,Shenzhen,True,True,2,580,2026-07-12,2026-07-03,3,1
3,U004,Diana,NaN,True,False,0,0,NaT,NaT,<NA>,0
4,U005,Evan,Chengdu,True,True,3,170,2026-07-04,2026-07-03,11,3
5,U006,Fang,Guangzhou,True,False,0,0,NaT,NaT,<NA>,0
6,U007,Gao,Wuhan,True,True,1,50,2026-07-11,NaT,4,1
7,U008,He,Xi'an,True,False,0,0,NaT,NaT,<NA>,0
8,U010,Ivy,Shanghai,True,False,0,0,NaT,NaT,<NA>,0
9,U999,NaN,NaN,False,True,2,450,2026-07-06,2026-07-05,9,2


### 9. paid_amount_rank

按照 `total_paid_amount` 从高到低排名。

规则：

```text
金额相同使用并列排名。
```

In [1131]:
# ======================================================
# SQL轨道（第十三步：支付金额排名）
# ======================================================

query_amount_rank = '''

    SELECT
        *,
        RANK()
        OVER(ORDER BY total_paid_amount DESC) AS paid_amount_rank
    FROM df_user_base
    ORDER BY paid_amount_rank, user_id
'''
df_user_base = duckdb.execute(query_amount_rank).fetchdf()
df_user_base

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid,max_consecutive_paid_days,paid_amount_rank
0,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12,2026-07-03,3,1,1
1,U001,Alice,Beijing,True,True,3,560.0,2026-07-10,2026-07-02,5,2,2
2,U999,NaN,NaN,False,True,2,450.0,2026-07-06,2026-07-05,9,2,3
3,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04,2026-07-03,11,3,4
4,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01,NaT,14,1,5
5,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11,NaT,4,1,6
6,U004,Diana,NaN,True,False,0,0.0,NaT,NaT,<NA>,0,7
7,U006,Fang,Guangzhou,True,False,0,0.0,NaT,NaT,<NA>,0,7
8,U008,He,Xi'an,True,False,0,0.0,NaT,NaT,<NA>,0,7
9,U010,Ivy,Shanghai,True,False,0,0.0,NaT,NaT,<NA>,0,7


### 10. user_tag

用户标签。

判断顺序从上到下：

| 条件 | user_tag |
|---|---|
| has_user_info = False | UNKNOWN_USER |
| has_paid_order = False | NO_PAID |
| total_paid_amount >= 500 and days_since_latest_paid <= 7 | HIGH_VALUE_ACTIVE |
| total_paid_amount >= 500 and days_since_latest_paid > 7 | HIGH_VALUE_SILENT |
| paid_order_count >= 3 | REPEAT_USER |
| 其他情况 | NORMAL_USER |


In [1132]:
# ======================================================
# SQL轨道（第十四步：user_tag）
# ======================================================

query_user_tag = '''

    SELECT
        *,
        CASE
            WHEN has_user_info = False THEN 'UNKNOWN_USER'
            WHEN has_paid_order = False THEN 'NO_PAID'
            WHEN  total_paid_amount >= 500 and days_since_latest_paid <= 7 THEN 'HIGH_VALUE_ACTIVE'
            WHEN total_paid_amount >= 500 and days_since_latest_paid > 7 THEN 'HIGH_VALUE_SILENT'
            WHEN paid_order_count >= 3 THEN 'REPEAT_USER'
            ELSE 'NORMAL_USER'
        END AS user_tag
    FROM df_user_base
    ORDER BY user_id

'''
df_user_paid_summary  = duckdb.execute(query_user_tag).fetchdf()
df_user_paid_summary 

,user_id,user_name,city,has_user_info,has_paid_order,paid_order_count,total_paid_amount,latest_paid_date,previous_paid_date,days_since_latest_paid,max_consecutive_paid_days,paid_amount_rank,user_tag
0,U001,Alice,Beijing,True,True,3,560.0,2026-07-10,2026-07-02,5,2,2,HIGH_VALUE_ACTIVE
1,U002,Bob,Shanghai,True,True,1,80.0,2026-07-01,NaT,14,1,5,NORMAL_USER
2,U003,Chen,Shenzhen,True,True,2,580.0,2026-07-12,2026-07-03,3,1,1,HIGH_VALUE_ACTIVE
3,U004,Diana,NaN,True,False,0,0.0,NaT,NaT,<NA>,0,7,NO_PAID
4,U005,Evan,Chengdu,True,True,3,170.0,2026-07-04,2026-07-03,11,3,4,REPEAT_USER
5,U006,Fang,Guangzhou,True,False,0,0.0,NaT,NaT,<NA>,0,7,NO_PAID
6,U007,Gao,Wuhan,True,True,1,50.0,2026-07-11,NaT,4,1,6,NORMAL_USER
7,U008,He,Xi'an,True,False,0,0.0,NaT,NaT,<NA>,0,7,NO_PAID
8,U010,Ivy,Shanghai,True,False,0,0.0,NaT,NaT,<NA>,0,7,NO_PAID
9,U999,NaN,NaN,False,True,2,450.0,2026-07-06,2026-07-05,9,2,3,UNKNOWN_USER
